# Data Science. Конспект лекций

Конспект курса «Основы Data Science» (учебный центр «Специалист.ru» при МГТУ им. Баумана).

## Этап 1. Формализация задачи

Заказчик предоставляет данные, связанные с некоторым **событием (фактом)** — например, «человек вернул или не вернул кредит».

Разметка данных — выделение в исходном массиве:
- **входных данных (X₁, X₂, …)** — свойств (контекста) события;
- **выходных данных (Y)** — результата события, который нужно научиться предсказывать.

Каждая строка полученной таблицы соответствует одному событию.

Далее нужно найти функцию **f**, связывающую входные данные с выходными:

$$Y = f(X_1, X_2, \dots)$$

Для **будущих событий** входные данные (X) уже известны, а выходные (Y) — ещё нет: применяя обученную функцию f к X, получаем предсказание Y.

![Формализация задачи: данные → разметка → таблица (входные X, выходные Y) → функция f → предсказание для будущих событий](images/pipeline_stage1.svg)

## Типы задач: регрессия и классификация

В зависимости от того, какая целевая переменная $Y$ (см. этап 1), задачи делят на два основных типа:

- **Регрессионная задача** — $Y$ непрерывная (числовая): нужно предсказать конкретное число. Примеры: `mpg` в `mtcars`, цена квартиры, температура.
- **Задача классификации** — $Y$ категориальная: нужно отнести объект к одному из классов. Примеры: «вернёт кредит / не вернёт», «спам / не спам», «кот / собака / птица».

От типа задачи зависит:
- вид функции/алгоритма — например, линейная регрессия для регрессии; логистическая регрессия, деревья классификации — для классификации;
- метрики оценки качества — для регрессии MAE, MSE, RMSE (раздел «Оценка модели»); для классификации — accuracy, precision, recall и другие, так как между классами нет числовой разности, которую можно усреднять.

## Этап 2. Выбор алгоритма подбора функции

На этом этапе выбирается **вид функции f** (алгоритм), связывающей входные данные (свойства события) с выходными, и подбираются её параметры — коэффициенты.

Примеры возможных видов функции:

$$Y = A x_1 + B x_2 + C \qquad \text{(линейная)}$$

$$Y = A x_1^2 + B x_1 x_2 + C x_2^2 + D \qquad \text{(квадратичная)}$$

Коэффициенты $A, B, C, D$ — параметры, которые нужно подобрать (обучить) по имеющимся данным.


**Выбор между простой и сложной функцией.** Более сложная функция (высокой степени) может пройти точно через все точки исходной выборки, а простая линейная регрессия — нет. Но это не значит, что сложная функция лучше: между точками выборки она может уходить в аномальные значения, которых там на самом деле нет. Линейная регрессия в такой ситуации часто даёт более надёжные предсказания на новых данных.

![Линейная регрессия против сложной функции: обе проходят близко к точкам выборки, но сложная функция сильно колеблется между ними](images/overfitting_example_v2.png)

## Выбросы

**Выброс (аномальное значение)** — точка данных, которая сильно выделяется на фоне общей закономерности остальных данных.

Выброс искажает обучение модели: если не убрать его из выборки, линия регрессии «тянется» к нему, отклоняясь от истинной закономерности в остальных данных.

На примере `mpg ~ wt`: синяя линия построена по всем точкам, включая выброс; зелёная — по тем же данным, но без выброса. Наклон и положение линии заметно меняются.

![Выброс смещает линию регрессии: регрессия с выбросом (синяя) отличается от регрессии без выброса (зелёная)](images/outlier_example.png)


**Как решить, убирать выброс или нет.** Само по себе отклонение точки от общей закономерности ещё не значит, что её нужно удалять — вывод делается по факту, а не заранее:
1. обучить модель с выбросом и без него (две версии);
2. сравнить качество предсказаний обеих версий (например, по MAE) на одних и тех же данных;
3. оставить ту версию, у которой ошибка предсказания меньше.


**Удаление выбросов (R)**: `boxplot()` сам находит выбросы (точки за пределами усов) и кладёт их в `$out`; дальше можно отфильтровать датафрейм, исключив строки с этими значениями.

```r
bp <- boxplot(mtcars$wt)
mtcars[!is.element(mtcars$wt, bp$out), ]
```

## Нормализация данных для нейросетей

Нейросети плохо работают с большими/разномасштабными входными числами и обучаются лучше, если входы нормализованы (например, в диапазон $[-1, 1]$ или $[0, 1]$). Причины:

1. **Насыщение функций активации.** Сигмоида и tanh «сплющивают» большие значения в почти константу, а производная в этой зоне почти нулевая — градиент, идущий назад при обратном распространении ошибки, становится крошечным (**vanishing gradient**), и веса перестают обновляться.
2. **Плохо обусловленный ландшафт оптимизации.** Если признаки сильно различаются по масштабу, поверхность функции потерь вытягивается в «овраг», и градиентный спуск в нём зигзагит вместо прямого пути к минимуму — сходится медленно или не сходится вовсе.
3. **Численная нестабильность.** Большие входы дают большие взвешенные суммы на нейронах и переполнение при вычислении $e^x$ (используется в sigmoid/softmax) — получаются `Inf`/`NaN`, которые ломают обучение.

Инициализация весов (Xavier/He) также рассчитана на входы с примерно единичной дисперсией — при ненормализованных входах эти схемы перестают работать как задумано.

## Визуализация (R)

Базовые функции визуализации в R на примере `mtcars`:

```r
plot(mtcars)
```
матрица диаграмм рассеяния — каждая переменная против каждой, удобно сразу увидеть все попарные зависимости.

![plot(mtcars)](images/r_plot_mtcars.png)

```r
plot(mtcars$wt)
```
для одного вектора — значение против порядкового номера наблюдения (индекса), без привязки к другой переменной.

![plot(mtcars$wt)](images/r_plot_wt_index.png)

```r
plot(mpg ~ wt, mtcars)
```
диаграмма рассеяния одной переменной против другой (формула) — видна форма зависимости между `wt` и `mpg`.

![plot(mpg ~ wt, mtcars)](images/r_plot_mpg_wt.png)

```r
hist(mtcars$wt)
```
гистограмма — распределение значений одной переменной по интервалам (частота).

![hist(mtcars$wt)](images/r_hist_wt.png)

```r
boxplot(mtcars$wt)
```
«ящик с усами» — медиана, квартили и выбросы одной переменной одним взглядом (точки за пределами усов — выбросы).

![boxplot(mtcars$wt)](images/r_boxplot_wt.png)

```r
boxplot(wt ~ cyl, mtcars)
```
тот же «ящик с усами», но отдельно по группам — удобно сравнивать распределение `wt` между разными значениями `cyl`.

![boxplot(wt ~ cyl, mtcars)](images/r_boxplot_wt_cyl.png)

## Категориальные данные

Если столбец закодирован числами, но по смыслу это не число, а категория (пол, статус, класс и т.п.) — такой столбец нужно переводить в категориальный тип, а не оставлять числовым: иначе модель будет считать эти значения количественно сравнимыми (например, что «2» в два раза больше «1»), хотя это просто разные категории.

Пример: файл `MyData2.txt`.

```
FirstName;LastName;Gender;Age;Salary
Evgeny;Onegin;1;26;100
Vladimir;Lensky;1;18;200
Tatyana;Larina;0;16;300
```

```r
my_data <- read.csv("/Users/ak/Code/ainots/specds/ds1/R/MyData2.txt", sep = ";")
my_data$Gender <- as.factor(my_data$Gender)
```

После этого `Gender` хранится как категориальный признак (`factor` с уровнями `0`/`1`), а не как число.

## Шкальные (порядковые) данные

**Шкальные данные** — промежуточный случай между категориальными и числовыми: значения — не числа, но между ними есть порядок, к ним применимо «больше/меньше» (например, уровень образования: низкий < средний < высокий). В отличие от обычных категорий (`factor`), порядок значений важен и должен сохраняться.

В R для этого — `factor(..., ordered = TRUE)` с явным указанием порядка уровней.

Пример: файл `MyData3.txt`.

```
FirstName;LastName;Education;Age;Salary
Evgeny;Onegin;High;26;100
Vladimir;Lensky;Medium;18;200
Tatyana;Larina;Low;16;300
```

```r
my_data <- read.csv("/Users/ak/Code/ainots/specds/ds1/R/MyData3.txt", sep = ";")
my_data$Education <- factor(my_data$Education, levels = c("Low", "Medium", "High"), ordered = TRUE)

my_data$Education[1] > my_data$Education[2] # TRUE — "High" > "Medium"
my_data[order(my_data$Education), ]         # сортировка по уровню, а не по алфавиту
```

## Конструирование признаков

**Конструирование признаков (feature engineering)** — создание новых, более информативных признаков на основе имеющихся сырых данных. Сырой признак сам по себе может быть менее полезен модели, чем то, что из него можно извлечь.

Примеры:
- **дата рождения** → возраст (1…100) → возрастная группа (1 — дети, 2 — подростки, 3 — молодёжь, 4 — зрелые, 5 — пожилые);
- **дата продажи** → день недели (будний/выходной), признак праздничного дня, сезон, время суток;
- **IP-адрес** → страна, город, провайдер, стаж аккаунта, признак VPN.

![Один сырой признак раскладывается на несколько новых, более информативных: дата рождения → возраст → возрастная группа; дата продажи → день недели/праздник/сезон/время суток; IP-адрес → страна/город/провайдер/стаж/VPN](images/feature_engineering.svg)

## Модель

**Модель** — результат объединения этапов I и II: способа разметки данных (формализации задачи) и выбранного алгоритма (вида функции f).

$$\text{Модель} = (\text{разметка данных}) + (\text{алгоритм подбора функции})$$

![Модель как объединение формализации задачи и выбора алгоритма](images/model_combination.svg)

## Этап 3. Обучение модели

**Обучение модели** — подбор коэффициентов функции (например, $A, B, C, D$ из этапа II) по имеющимся данным.

Процесс обучения — долгий, сложный, итерационный.

## Проверка модели

**Проверка модели** — оценка того, насколько предсказания $f(X)$ совпадают с реальными значениями $Y$.

Данные для этого делят на:
- **обучающую выборку** — по ней подбираются коэффициенты функции (этап III);
- **тестовую (проверочную) выборку** — данные, которые модель не видела при обучении; на них сравнивают предсказанное значение с реальным и считают ошибку/качество модели.

Проверка обязательно проводится на данных, **отличных** от тех, на которых модель обучалась — иначе оценка ошибки получится заниженной и необъективной.


Пример (R, датасет `mtcars`): две модели предсказывают `mpg`, сравниваем среднюю абсолютную ошибку (MAE). Для простоты пример считает ошибку на тех же данных, на которых обучались — на практике для этого нужна отдельная тестовая выборка.

```r
M <- mtcars

model1 <- lm(mpg ~ wt, M)
p1 <- predict(model1, M)

model2 <- lm(mpg ~ wt + hp + cyl, M)
p2 <- predict(model2, M)

Result <- data.frame(M$mpg, p1, p2) # сводим предсказания и реальные значения в одну таблицу

mean(abs(M$mpg - p1)) # Mean ABS Error, model1
mean(abs(M$mpg - p2)) # Mean ABS Error, model2
```

У `model2` (три предиктора: `wt`, `hp`, `cyl`) ошибка меньше, чем у `model1` (один предиктор `wt`): MAE ≈ 1.84 против MAE ≈ 2.34.


Из проверенных моделей выбирается та, что удовлетворяет требованиям заказчика — например, требованию, что ошибка предсказания должна быть меньше заданного числа.


**Полный цикл** (на примере `mtcars`): данные размечаются на X (признаки) и Y (целевая переменная), делятся на обучающую (70%) и контрольную (30%) выборки. На обучающей выборке и выбранном алгоритме получаем обученную модель; её оценивают на контрольной выборке по заранее заданным критериям качества. Если критериям не соответствует — цикл повторяется (другой алгоритм, другая разметка и т.д.), если соответствует — модель идёт в эксплуатацию.

![Полный цикл: разметка данных → обучающая/контрольная выборка (70/30) → обучение → обученная модель → оценка модели (по критериям качества) → эксплуатация, либо повтор цикла](images/train_eval_cycle.svg)

## Оценка модели (метрики ошибки)

На контрольной выборке для каждого наблюдения сравнивают реальное значение $y_i$ (красная точка) с предсказанием модели $p_i$ (зелёная точка); их разность — величина ошибки на этом наблюдении.

Чтобы получить одну числовую оценку качества модели по всей контрольной выборке, ошибки отдельных наблюдений сводят в метрику:

- **MAE** (Mean Absolute Error) $= mean(|y_i - p_i|)$ — средняя абсолютная ошибка;
- **MSE** (Mean Squared Error) $= mean((y_i - p_i)^2)$ — средняя квадратичная ошибка;
- **RMSE** (Root MSE) $= \sqrt{MSE}$ — корень из MSE, возвращает ошибку в исходные единицы измерения $y$ (у MSE единицы — квадратные).

MAE и MSE отличаются не только единицами измерения, но и тем, как штрафуются большие ошибки:
- в MAE каждая единица ошибки увеличивает метрику одинаково (линейно, как $|x|$);
- в MSE ошибка возводится в квадрат (как $x^2$), поэтому маленькие ошибки почти не штрафуются, а большие ошибки и выбросы штрафуются значительно сильнее.

![Оценка модели: реальные (красные) и предсказанные (зелёные) значения на контрольной выборке, величина ошибки между ними; справа — сравнение формы x² и |x|, показывающее, что MSE резче штрафует большие ошибки, чем MAE](images/error_metrics.svg)

## Оценка ошибки: регрессия и классификация

Способ оценки ошибки напрямую зависит от типа задачи (см. «Типы задач: регрессия и классификация»).

**Регрессия.** $Y$ — непрерывная величина, поэтому ошибка на одном примере — это расстояние между реальным и предсказанным числом: $|y_i - p_i|$. Чем ближе предсказание к реальному числу, тем меньше ошибка. На этом строятся метрики MAE, MSE, RMSE (раздел «Оценка модели»).

**Классификация.** $Y$ — категориальная величина, классы не упорядочены и не являются числами, поэтому «расстояния» между ними нет. Предсказание либо совпадает с истинным классом (**верный ответ**), либо нет (**ошибка**) — промежуточных вариантов не бывает.

Отсюда базовая метрика классификации — доля правильных ответов:

$$accuracy = \frac{\text{число верных ответов}}{\text{общее число примеров}}$$

Доля ошибок (error rate) $= 1 - accuracy$.

![Регрессия: ошибка — расстояние |y − p| между реальным и предсказанным числом. Классификация: ошибка — несовпадение предсказанного класса с истинным (верный ответ / ошибка), расстояния между классами нет](images/error_regression_vs_classification.svg)

## Порог уверенности для вероятностного классификатора

Классификатор, который выдаёт не единственный класс, а вектор вероятностей по всем классам (например, нейросеть из MNIST-примера — раздел «Бэггинг/Стекинг/Бустинг» выше про ансамбли, а тут отдельная сеть), считает accuracy (см. «Оценка ошибки: регрессия и классификация») не напрямую: сначала нужно решить, что считать «ответом» модели.

**Порог верного ответа (`thresh`)** — вероятность, выше которой класс считается ответом модели. Для каждого примера возможны три исхода:
- ровно один класс выше порога, и он совпадает с истинной меткой → верный ответ;
- ровно один класс выше порога, но он не совпадает с истинной меткой → ошибка;
- ни один класс не превысил порог, либо превысили сразу несколько → ответ неоднозначный (модель «не уверена») → тоже ошибка.

Accuracy по-прежнему считается как число верных ответов относительно общего числа примеров, но порог можно крутить: выше `thresh` — модель «осторожнее» (меньше уверенных, но неверных ответов, больше неоднозначных случаев); ниже `thresh` — почти всегда есть единственный ответ, но выше риск засчитать как верное малоуверенное предсказание.

Пример (R, MNIST, файл `Nums4.R`): для каждого тестового примера берём вектор вероятностей `result[j, ]`, оставляем классы выше порога; если ровно один такой класс и он совпадает с истинной меткой — засчитываем верный ответ.

```r
thresh <- 0.7
correct <- 0
for (j in 1:mnist$test$n) {
  lamps <- result[j, ]
  ans <- (0:9)[lamps > thresh]   # классы, прошедшие порог

  if (length(ans) != 1) {
    next                         # неоднозначный ответ — тоже ошибка
  }

  if (ans == mnist$test$y[j]) {
    correct <- correct + 1
  }
}

accuracy <- correct / mnist$test$n
```

## Матрица сопряжённости (Confusion Matrix)

Accuracy — одно число, но оно прячет **тип** ошибки. Для бинарной классификации (пример: «болен COVID (1) / здоров (0)») ошибка бывает двух разных видов с разной ценой последствий — это видно только если разложить ответы на 4 группы.

**Матрица сопряжённости** — таблица 2×2: строки — истинный класс, столбцы — наш прогноз.

| | Прогноз = 1 | Прогноз = 0 |
|---|---|---|
| **Истинный класс = 1** (болен) | **TP** (True Positive) — верно определили больного | **FN** (False Negative, ложноотрицательный) — на самом деле болен, но модель сказала «здоров» |
| **Истинный класс = 0** (здоров) | **FP** (False Positive, ложноположительный) — на самом деле здоров, но модель сказала «болен» | **TN** (True Negative) — верно определили здорового |

- TP и TN — верные ответы, из них и складывается accuracy: $accuracy = \dfrac{TP + TN}{TP + TN + FP + FN}$.
- **FN (ложноотрицательный)** — обычно самая опасная ошибка для диагностики: заразного/больного человека отпустили как здорового.
- **FP (ложноположительный)** — ложная тревога: здорового приняли за больного — неприятно, но менее опасно, чем пропустить болезнь.

**Связь с порогом** (см. «Порог уверенности для вероятностного классификатора»): меняя `thresh`, можно сознательно двигать баланс между FN и FP. Например, для теста на COVID часто выгоднее занизить порог — это увеличит FP (больше ложных тревог), зато уменьшит FN (меньше пропущенных больных), потому что цена пропустить больного выше цены ложной тревоги.

## Сведение многоклассовой классификации к бинарной

Матрица сопряжённости (TP/FP/FN/TN) по своей природе — понятие для **бинарной** классификации (два класса). Но, например, в задаче MNIST классов не два, а десять (цифры 0–9) — как тогда применить эту матрицу?

**Приём (one-vs-rest, «один против всех»)**: для каждого класса по очереди этот класс объявляется «положительным» (1), а все остальные классы объединяются в один общий «отрицательный» (0). Так одна задача на 10 классов превращается в 10 отдельных бинарных задач — по одной на класс, и для каждой уже можно строить обычную матрицу сопряжённости.

Пример — оценка того, как модель распознаёт цифру `0`:
- положительный класс — «это цифра 0»;
- отрицательный класс — «это любая из цифр 1, 2, 3, …, 9» (все девять остальных цифр объединяются в одну группу).

| | Прогноз = 0 | Прогноз ≠ 0 |
|---|---|---|
| **Истинный класс = 0** | ИП — истинно положительный (TP) | ЛО — ложноотрицательный (FN) |
| **Истинный класс ≠ 0** | ЛП — ложноположительный (FP) | ИО — истинно отрицательный (TN) |

Чтобы оценить модель по всем 10 цифрам, эту процедуру повторяют отдельно для каждой (0-vs-rest, 1-vs-rest, …, 9-vs-rest) — получается 10 матриц сопряжённости, по одной на класс.

## Точность и полнота (Precision, Recall)

Accuracy может вводить в заблуждение, особенно при несбалансированных классах. Из TP/FP/FN/TN (матрица сопряжённости) считают ещё две метрики, которые смотрят на ошибки отдельно с двух разных сторон.

**Точность (Precision)**:
$$Precision = \dfrac{ИП}{ИП + ЛП} = \dfrac{TP}{TP + FP}$$
Какую долю объектов, распознанных моделью как объекты положительного класса, мы предсказали верно. Иными словами — насколько можно доверять ответу «1» модели.

**Полнота (Recall, True Positive Rate)**:
$$Recall = \dfrac{ИП}{ИП + ЛО} = \dfrac{TP}{TP + FN}$$
Какую долю объектов, реально относящихся к положительному классу, мы предсказали верно. Иными словами — сколько из всех настоящих «1» модель нашла, а не пропустила.

Между ними обычно компромисс, регулируемый порогом `thresh` (см. «Порог уверенности…»): выше порог → модель реже отвечает «1» → Precision растёт, а Recall падает (больше пропущенных, FN растёт); ниже порог — наоборот.

## F-мера (F-score)

Число, объединяющее Precision и Recall в одну метрику — их **среднее гармоническое** (не обычное среднее арифметическое):

$$F = 2 \cdot \dfrac{Precision \cdot Recall}{Precision + Recall}$$

Гармоническое среднее, в отличие от арифметического, сильно штрафует случай, когда одна из двух метрик низкая, даже если другая близка к максимуму — F-мера получается высокой, только если высоки **обе** метрики одновременно. Поэтому F-мера удобна как единственное число для сравнения моделей или порогов, когда важны и Precision, и Recall сразу, а не какая-то одна из них.

## ROC-кривая

Все метрики выше (accuracy, Precision, Recall, F-мера) считаются **при одном конкретном пороге** `thresh`. Чтобы оценить качество вероятностного классификатора сразу по всем порогам, используют ROC-кривую (Receiver Operating Characteristic).

Строится она так: перебирают все возможные значения порога от 1 до 0, и для каждого считают две величины из матрицы сопряжённости:
- **TPR** (Recall, доля истинноположительных) — по оси Y: $TPR = \dfrac{ИП}{ИП + ЛО}$;
- **FPR** (доля ложноположительных) — по оси X: $FPR = \dfrac{ЛП}{ЛП + ИО}$.

Каждый порог даёт одну точку (FPR, TPR); соединив точки для всех порогов, получаем кривую от (0,0) — «никого не отмечаем положительным» (`thresh = 1`) — до (1,1) — «отмечаем положительным всех» (`thresh = 0`).

![ROC-кривая: TPR против FPR для всех порогов, диагональ — случайное угадывание, закрашенная площадь под кривой — AUC](images/roc_curve.svg)

- **Диагональ** (0,0)→(1,1) — эталон случайного угадывания: на любом пороге столько же истинных срабатываний, сколько и ложных.
- Чем **ближе кривая к левому верхнему углу**, тем лучше классификатор: высокий TPR (не пропускает положительные) при низком FPR (мало ложных тревог) одновременно.

## AUC (Area Under Curve)

**AUC** — площадь под ROC-кривой, одно число от 0 до 1:
- AUC = 1 — идеальный классификатор;
- AUC = 0.5 — не лучше случайного угадывания (совпадает с диагональю);
- AUC < 0.5 — хуже случайного (модель систематически ошибается «наоборот»).

Главное преимущество AUC — она **не зависит от выбора порога**: сравнивает модели сразу по всей ROC-кривой, а не в одной конкретной точке. Это удобно, когда порог ещё не выбран или должен настраиваться отдельно под задачу (см. «Порог уверенности…», «Связь с порогом» в Precision/Recall).

![Сравнение ROC-кривых классификаторов разного качества: отличный (AUC≈0.91), средний (AUC≈0.72), случайный (AUC=0.5)](images/roc_auc_comparison.svg)

## Перекрёстная оценка

Одно случайное разбиение на обучающую и контрольную выборку может дать нерепрезентативную оценку — результат зависит от того, какие именно строки попали в контрольную выборку.

**Перекрёстная оценка (кросс-валидация)**: данные делят на несколько равных блоков (например, 6). По очереди каждый блок становится контрольной (проверочной) выборкой, а остальные блоки — обучающей; для каждого такого разбиения обучается своя модель и считается метрика ошибки (например, MAE).

В результате получается несколько значений метрики (по одному на модель) — их **разброс** показывает, насколько стабильно модель работает на разных данных.

![Перекрёстная оценка: данные разбиты на 6 блоков, по очереди каждый блок — проверочный, остальные — обучающие; MAE считается для каждого разбиения, разброс MAE между разбиениями — оценка стабильности модели](images/cross_validation.svg)

## Бэггинг (Random Forest)

**Бэггинг** (bootstrap aggregating) — способ построения ансамбля моделей:
1. алгоритм (вид модели) — **одинаковый** для всех моделей;
2. обучающие выборки — **разные** (например, 100 случайных подвыборок одних и тех же исходных данных).

Каждая из моделей обучается на своей выборке и делает свой прогноз для одного и того же нового объекта: $y_1, y_2, \dots, y_{100}$.

Итоговый прогноз — среднее значение всех индивидуальных прогнозов:

$$\hat{Y} = \frac{1}{100}\sum_{i=1}^{100} y_i$$

**Random Forest** — конкретная реализация бэггинга, где базовый алгоритм — дерево решений.

![Бэггинг: 100 разных подвыборок → одинаковый алгоритм → 100 моделей → 100 прогнозов для одного объекта → среднее значение прогноза](images/bagging_random_forest.svg)

## Стекинг

**Стекинг** — ещё один способ построения ансамбля моделей, устроенный противоположно бэггингу:
1. алгоритмы — **разные** (например, дерево решений, линейная регрессия, случайный лес);
2. обучающая выборка — **одна и та же** для всех алгоритмов;
3. используется **дополнительный (итоговый/решающий) алгоритм**, который учится комбинировать предсказания базовых моделей.

Каждый из базовых алгоритмов делает свой прогноз ($y_1, y_2, y_3, y_4$) на одной и той же выборке. Эти прогнозы становятся **входными данными** для итогового алгоритма, который и выдаёт финальный прогноз $y$.

В отличие от бэггинга (где результат — простое усреднение одинаковых моделей на разных выборках), в стекинге комбинирование предсказаний разных моделей само является отдельной обучаемой задачей.

![Стекинг: одна выборка → несколько разных алгоритмов → их прогнозы y1..y4 → итоговый (решающий) алгоритм → финальный прогноз y](images/stacking.svg)

## Бустинг

**Бустинг** — ещё один способ построения ансамбля, но, в отличие от бэггинга и стекинга, **последовательный**:
1. алгоритм — **один и тот же** на каждой итерации;
2. вместо параллельного обучения на разных выборках или разных алгоритмах, модель обучается **много раз подряд**, и каждая следующая итерация подбирает выборку так, чтобы лучше предсказывать те объекты, на которых предыдущая модель ошибалась больше всего.

Цикл одной итерации: обучение → модель → тестирование → находим объекты с наибольшей ошибкой → увеличиваем их вес в обучающей выборке → снова обучение…

Каждая следующая модель «исправляет» ошибки предыдущей, постепенно улучшая общий результат.

![Бустинг: обучение → модель → тестирование → поиск объектов с наибольшей ошибкой → увеличение их веса → снова обучение (один и тот же алгоритм, много итераций подряд)](images/boosting.svg)

## Этап 4. Эксплуатация модели

**Обученная модель** — готовая формула (функция f с подобранными коэффициентами).

Эксплуатация модели — применение готовой формулы к новым (будущим) входным данным для получения результата (предсказания).

## Общая схема (со слайда)

![Общая схема этапов I-IV](images/overview_screenshot.png)